In [ ]:
import asyncio
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from scipy import signal
from scipy.signal import welch
from idun_guardian_sdk import GuardianClient

# --- KONFIGURATION ---
MY_API_TOKEN = "idun_MWSQ4pkewAGNz8wwYzw_NsweXihLC8tIcFzah8vqqys4Nc-ALzjfTwl2"
fs = 250 
WINDOW_SECONDS = 5 
BUFFER_SIZE = fs * WINDOW_SECONDS

# --- FREQUENZBÄNDER ---
bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma": (30, 40)
}
bandslist = ["delta", "theta", "alpha", "beta", "gamma"]

# --- FILTER SETUP ---
sos_bp = signal.butter(N=4, Wn=[1, 40], btype="bandpass", fs=fs, output="sos")
b_notch, a_notch = signal.iirnotch(50, Q=30, fs=fs)

def apply_window_filters(data_array):
    """Wendet die Filter auf das aktuelle Zeitfenster an."""
    if len(data_array) < fs:
        return data_array
    
    # In float umwandeln für präzise Berechnungen
    data_array = np.asfarray(data_array)
    
    # Bandpass & Notch
    filtered = signal.sosfilt(sos_bp, data_array)
    filtered = signal.filtfilt(b_notch, a_notch, filtered)
    
    # DC Offset entfernen
    filtered = filtered - np.mean(filtered)
    return filtered

In [ ]:
# --- DATA STORAGE ---
class LiveData:
    def __init__(self):
        self.timestamps = []
        self.ch1 = []
        self.is_running = True

live_buffer = LiveData()

# --- CALLBACK FUNKTION ---
def handle_live_data(event):
    if type(event).__name__ == "LiveInsightsEvent":
        # Prüfen ob raw_eeg Daten vorhanden sind
        if hasattr(event, 'message') and 'raw_eeg' in event.message:
            samples = event.message['raw_eeg']
            
            for sample in samples:
                # Hier ziehen wir die exakten Keys aus deinem Output
                val = sample.get('ch1', 0)
                ts = sample.get('timestamp', 0)
                
                live_buffer.ch1.append(val)
                live_buffer.timestamps.append(ts)
            
            # Buffer begrenzen (Sliding Window)
            if len(live_buffer.ch1) > BUFFER_SIZE:
                live_buffer.ch1 = live_buffer.ch1[-BUFFER_SIZE:]
                live_buffer.timestamps = live_buffer.timestamps[-BUFFER_SIZE:]

In [ ]:
async def live_plot_task():
    """Aktualisiert den PSD-Plot mit eingefärbten Frequenzbändern."""
    # Wir brauchen nperseg für Welch. Offline hattest du fs*4. 
    # Dafür muss unser Buffer mindestens 4 Sekunden lang sein.
    nperseg_val = fs * 4 
    
    fig, ax = plt.subplots(figsize=(10, 5))
    plt.ion()
    
    while live_buffer.is_running:
        # Wir warten, bis der Buffer voll genug für die Welch-Methode ist
        if len(live_buffer.ch1) >= nperseg_val:
            # Kopie der aktuellen Daten ziehen
            data = np.array(live_buffer.ch1)
            
            # 1. Filtern
            filtered_data = apply_window_filters(data)
            
            # 2. PSD Berechnen (wie in deinem Code)
            freqs, psd = welch(
                filtered_data,
                fs=fs,
                nperseg=nperseg_val,
                noverlap=fs * 2
            )
            
            # 3. Plotten
            ax.clear()
            ax.semilogy(freqs, psd, color='black', linewidth=1)
            
            # Bänder einfärben
            for band, (f_low, f_high) in bands.items():
                idx_band = np.logical_and(freqs >= f_low, freqs <= f_high)
                ax.fill_between(
                    freqs[idx_band], 
                    psd[idx_band], 
                    alpha=0.5, 
                    label=f"{band} ({f_low}-{f_high} Hz)"
                )
            
            # Plot formatieren
            ax.set_xlabel("Frequency (Hz)")
            ax.set_ylabel("Power Spectral Density (mV²/Hz)")
            ax.set_title("Live EEG Power Spectrum")
            ax.grid(True, alpha=0.3)
            ax.set_ylim([0.05, 50])
            ax.set_xlim([0.5, 35])
            ax.legend(loc='upper right')
            
            # Jupyter UI Update
            clear_output(wait=True)
            display(fig)
            
        await asyncio.sleep(0.5) # Update-Rate (alle 500ms)
        
    plt.close(fig)

In [ ]:
async def main():
    client = GuardianClient(api_token=MY_API_TOKEN, debug=False)
    
    # Subscribe (wir brauchen hier nur raw_eeg für unsere eigene Berechnung)
    client.subscribe_live_insights(raw_eeg=True, handler=handle_live_data)
    
    print("Verbindung wird aufgebaut... Sammle erste 5 Sekunden Daten...")
    
    plot_task = asyncio.create_task(live_plot_task())
    
    try:
        # Aufnahme starten (z.B. für 3 Minuten = 180 Sekunden)
        await client.start_recording(recording_timer=180, led_sleep=False, calc_latency=False)
    except Exception as e:
        print(f"Fehler: {e}")
    finally:
        live_buffer.is_running = False
        await plot_task
        print("Aufnahme beendet.")

# Zelle ausführen
await main()